In [50]:
from re import L
import h5py
import torch
import glob
import sys
import os 
import pickle
import numpy as np
from pathlib import Path

In [51]:
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 

In [52]:
word_list = list(class_map.keys())

In [53]:
import nltk
from nltk.corpus import wordnet

In [54]:
# Download WordNet if not already downloaded
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

def remove_tenses(words):
	# Initialize a lemmatizer
	lemmatizer = nltk.stem.WordNetLemmatizer()

	# Define a function to get the part of speech for each word
	def get_wordnet_pos(word):
		tag = nltk.pos_tag([word])[0][1][0].upper()
		tag_dict = {"J": wordnet.ADJ, 
					"N": wordnet.NOUN,
					"V": wordnet.VERB, 
					"R": wordnet.ADV}
		return tag_dict.get(tag, wordnet.NOUN)  # Default to noun if not found

	# Lemmatize each word based on its part of speech and store in a set to remove duplicates
	base_words = set()
	for word in words:
		base_word = lemmatizer.lemmatize(word)#, pos=get_wordnet_pos(word))
		base_words.add(base_word)

	return list(base_words)

[nltk_data] Downloading package wordnet to /home/imgriff/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/imgriff/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [55]:
word_list = [word for word in word_list if "'" not in word]
len(word_list)

784

In [56]:
base_words = remove_tenses(word_list)
len(base_words)

742

In [57]:
shorter_words = [word for word in base_words if 4 <= len(word) <= 7]

In [58]:
len(shorter_words)

551

In [59]:
shorter_words.remove('centre')

In [60]:
shorter_words.sort()
print(shorter_words)

['about', 'above', 'access', 'across', 'action', 'active', 'added', 'africa', 'after', 'again', 'against', 'airport', 'album', 'allowed', 'almost', 'alone', 'along', 'already', 'always', 'america', 'among', 'ancient', 'animal', 'another', 'appear', 'appears', 'area', 'around', 'artist', 'asked', 'attack', 'author', 'award', 'based', 'battle', 'beach', 'became', 'because', 'become', 'before', 'began', 'behind', 'being', 'believe', 'below', 'better', 'between', 'bird', 'black', 'blood', 'board', 'book', 'border', 'bridge', 'bright', 'bring', 'british', 'brother', 'brought', 'brown', 'built', 'buried', 'called', 'canada', 'cannot', 'capital', 'career', 'carried', 'case', 'cause', 'caused', 'center', 'central', 'certain', 'change', 'changed', 'charge', 'charles', 'chief', 'child', 'china', 'chinese', 'church', 'city', 'civil', 'class', 'clear', 'close', 'closed', 'coach', 'coast', 'college', 'color', 'come', 'coming', 'common', 'company', 'complex', 'concept', 'control', 'could', 'council'

In [61]:
# make sure shorter_words are actually in word_list
short_word_set = set(shorter_words)
word_list_set = set(word_list)
included_words = short_word_set.intersection(word_list_set)
print(len(included_words))


# find words in shorter_words that are not in word_list
missing_words = short_word_set - word_list_set
print(len(missing_words.intersection(short_word_set)))

## Make sure included words are actually in the word_list
print(len(included_words.intersection(word_list_set)))
for word in included_words:
	assert word in word_list, f"{word} not in word_list"

505
45
505


In [62]:
included_words_list = list(included_words)
for word in included_words:
	# if word == 'centre' or word == 'center':
	#     print(word)
	for prefix in ['s', 'ed', 'er', 'd']:
		if word + prefix in included_words:
			print(word)
			print(word + prefix)
			included_words_list.remove(word + prefix)

cause
caused
change
changed
appear
appears
offer
offered
young
younger
other
others
force
forced
place
placed
small
smaller
close
closed
start
started
cover
covered
found
founded


In [63]:
len(included_words_list)

492

In [64]:
included_words_list.sort()
print(included_words_list)

['about', 'above', 'access', 'across', 'action', 'active', 'added', 'africa', 'after', 'again', 'against', 'airport', 'album', 'allowed', 'almost', 'alone', 'along', 'already', 'always', 'america', 'among', 'ancient', 'another', 'appear', 'around', 'artist', 'asked', 'attack', 'author', 'award', 'based', 'battle', 'beach', 'became', 'because', 'become', 'before', 'began', 'behind', 'being', 'believe', 'below', 'better', 'between', 'black', 'blood', 'board', 'border', 'bridge', 'bright', 'bring', 'british', 'brother', 'brought', 'brown', 'built', 'buried', 'called', 'canada', 'cannot', 'capital', 'career', 'carried', 'cause', 'center', 'central', 'certain', 'change', 'charge', 'charles', 'chief', 'child', 'china', 'chinese', 'church', 'civil', 'class', 'clear', 'close', 'coach', 'coast', 'college', 'color', 'coming', 'common', 'company', 'complex', 'concept', 'control', 'could', 'council', 'country', 'county', 'couple', 'course', 'court', 'cover', 'created', 'creek', 'cross', 'culture',

In [65]:
## Manually remove specific words
included_words_list.remove('there')
included_words_list.remove('killed')
included_words_list.remove('death')
included_words_list.remove('blood')
# included_words_list.remove('covered')


In [66]:

len(included_words_list)

488

In [67]:
with open("word_list_for_binaural_sentences.pkl", "wb") as output:
	pickle.dump(included_words_list, output)

## Use OpenAI GPT to generate text for target words

Need two target sentences per word

Need two cue sentences use non-taret words, but have the same general length 

In [21]:
from io import StringIO
# import typing_extensions 
from openai import OpenAI
from api_util.openai_api_key import API_KEY
from pathlib import Path


gen_path = Path('generated_experiment_stimuli/attn_epmnt_text_promps_v00/')
# make directories
gen_path.mkdir(parents=True, exist_ok=True)


client = OpenAI(api_key=API_KEY)


#### If making all words

In [2]:
import pickle
with open("word_list_for_binaural_sentences.pkl", "rb") as f:
	target_word_list =  pickle.load(f)

### If using regen list

In [51]:
import pickle
# with open("/om2/user/rphess/Auditory-Attention/regen_words.pkl", "rb") as f:
# 	target_word_list =  pickle.load(f)

with open(gen_path / "regen_words.pkl", "rb") as f:
	target_word_list =  pickle.load(f)

# is dict of words and bad sentences. Just regenerate from words 

target_word_list = list(target_word_list.keys())

In [52]:
len(target_word_list)

3

In [53]:
import string 

def package_gen_str_tsv(gen_str):
    results = []
    for row in gen_str.split("```")[1].split('\n'):
        if row == '':
            continue 
        if 'WORD' in row:
            continue
        word, sentence1, sentence2 = row.split('\t')
        # strip all punctuation from word using string module 
        word = word.translate(str.maketrans('', '', string.punctuation))

        results.append(dict(word=word, sentence1=sentence1, sentence2=sentence2))
    df_results = pd.DataFrame.from_records(results)
    return df_results

### Generation will take rougly 10 minutes

In [54]:
import json 
import pandas as pd


completion_dicts = []
# n_chunks = 4 if len(target_word_list) > 100 else 2
chunk_size = len(target_word_list)//1 

for i in range(0, len(target_word_list), chunk_size):
	sub_list = target_word_list[i:i+chunk_size]
	# sub_list = target_word_list[200:204]
	messages = [
		{"role":"system", "content": f"I will provide you a list of words in a python list format. You will return a list of two sentences per word in as a TSV in the order of WORD, SENTENCE 1, SENTENCE 2." \
		"You will use the word in its exact form in the middle of each sentence with 5 or more words. Do not reuse words from this list in other sentences. They should only occur in their own sentences." \
		"Make sure these words in the list appear only in the middle of their sentence. Make sure there is at least two words before and two words after the target word."},
						# "Return the results with each word as a key and a  containing both sentences as the corresponding value." 

		
		# {"role":"system", "content": f"Here is the list of words you will be working with: {target_word_list}"}
		# {"role":"system", "content": f"Do not reuse words from this list in other sentences. They should only occur in their own sentences. Make sure these words in the list appear only in the middle of their sentence."},

		
	# {"role":"system", "content": "An example of a valid sentence given the target word 'around' would be {'around': ['The dog ran around the tree.', 'Tom went home around five this evening.']}"},
	# {"role":"system", "content": "An example of a valid sentence given the target word 'africa' would be {'africa': ['The trip to Africa is long.', 'The animals in Africa are unique']}"},
	{"role":"user", "content": f"Write two sentences for each word in this list of target words {sub_list}. Please return the data only, do not include a response message."},
	]

	response = client.chat.completions.create(
	model="gpt-4-turbo",
	messages=messages,
	)
	# decode JSON returned from response
	# if json in first 10 elements of string, remove it
	returned_sentences_str = response.choices[0].message.content
	try:
		data = pd.read_csv(StringIO(returned_sentences_str), sep='\t',  header=None)
		data.columns = ['word', 'sentence1', 'sentence2']
		completion_dicts.append(data)
	except:
		print("Error with response, attept repackage string ")
		completion_dicts.append(package_gen_str_tsv(returned_sentences_str))
	# update completion_dicts with the generated sentences
	# completion_dicts = {**completion_dicts, **generated_sentences}

# print(response.choices[0].message.content)



In [55]:
new_dfs = pd.concat(completion_dicts, axis=0, ignore_index=True)


In [56]:
new_dfs

,word,sentence1,sentence2
0,spent,He had spent several hours contemplating the f...,They spent the weekend renovating their old ho...
1,worked,She eagerly worked through the complex problem.,The team worked together to achieve their goal.
2,young,The young artist gained recognition quickly.,They adopted a young puppy from the shelter.


In [57]:
dict_to_share_regen_words = {item['word']: [item['sentence1'], item['sentence2']] for item in new_dfs.to_dict(orient='records')}

with open(gen_path / "next_pass_regen_words.pkl", "wb") as output:
	pickle.dump(dict_to_share_regen_words, output)

In [43]:
len(dict_to_share_regen_words)

8

## New ones from poor generations

In [27]:
len(completion_dicts)

47

In [132]:
for ix, item in enumerate(completion_dicts):
    if isinstance(item, str):
        print(ix)

1
2
3


In [167]:
new_dfs = []
for batch in completion_dicts:
    if isinstance(batch, str):
        new_dfs.append(package_gen_str_tsv(batch))
    else:
        new_dfs.append(batch)

new_dfs = pd.concat(new_dfs, axis=0, ignore_index=True)

In [174]:
dict_to_share_regen_words = {item['word']: [item['sentence1'], item['sentence2']] for item in new_dfs.to_dict(orient='records')}


In [175]:
len(dict_to_share_regen_words)

316

In [178]:

# write dict_to_share as pickle 
with open(gen_path / "second_pass_regen_words.pkl", "wb") as output:
    pickle.dump(dict_to_share_regen_words, output)

In [169]:
set(new_dfs['word'].to_list()) == set(target_word_list)

True

#### Import filtered generations 
filtering done in notebook filter_generated_sentences.ipynb

In [ ]:
with open(gen_path / 'final_list_of_target_transcripts.pkl', 'rb') as f:
	transcripts_to_generate = pickle.load(f)

# old ones 

In [78]:
results_0 = []
for row in completion_dicts[0].split("```")[1].split('\n')[2:]:
	word = row.split(',')[0]
	sentences = row[len(word):]
	# split sentences at sentence ending punctuation 
	if '?' in sentences:
		sentence1, sentence2 = sentences.split('?,')
	elif '.' in sentences:
		sentence1, sentence2 = sentences.split('.,')
	# strip leading , from sentence1
	sentence1 = sentence1[1:]

	# sentence1, sentence2 = sentences.split('.,')
	results_0.append(dict(word=word, sentence1=sentence1, sentence2=sentence2))
	
df_results_0 = pd.DataFrame.from_records(results_0)

In [92]:
results_1 = []
import re

for row in completion_dicts[1].split("```")[1].split('\n')[2:]:
	word = row.split(',')[0]
	sentences = row[len(word)+1:]
	# split sentences at sentence ending punctuation 
	if '?' in sentences and sentences[-1] != '?':
		sentence1, sentence2 = sentences.split('?,')
	elif '"' in sentences:
		sentence1, sentence2 = sentences.split('",')

	elif '.,' in sentences:
		sentence1, sentence2 = sentences.split('.,')
	# strip leading , from sentence1
	# sentence1 = sentence1[1:]

	# sentence1, sentence2 = sentences.split('.,')
	results_1.append(dict(word=word, sentence1=sentence1, sentence2=sentence2))
	
df_results_1 = pd.DataFrame.from_records(results_1)

In [94]:
merged_df = pd.concat([df_results_0, df_results_1, completion_dicts[2].iloc[1:]], axis=0, ignore_index=True)

In [105]:
dict_to_share = {item['word']: [item['sentence1'], item['sentence2']] for item in merged_df.to_dict(orient='records')}


In [106]:
# write dict_to_share as pickle 
with open("eg_binaural_sentences_dict.pkl", "wb") as output:
    pickle.dump(dict_to_share, output)

In [12]:
import pandas as pd

data = pd.read_csv(StringIO(returned_sentences_str), header=None)
data.columns = ['word', 'sentence1', 'sentence2']
data


,word,sentence1,sentence2
0,island,The treasures were hidden somewhere on the isl...,We visited a small island off the coast that w...
1,issue,The company finally addressed the issue after ...,She had an issue with the way the news was pre...
2,italian,We went to an Italian restaurant that had the ...,My friend learned how to cook authentic Italia...
3,itself,"The novel itself is a masterpiece, but the ada...","The problem resolved itself overnight, much to..."
